# Clase 121 — Custom training loops

Escribimos un **training loop manual** en TF con `GradientTape`: control total
sobre cada paso (útil para GANs, RL, multi-optimizer, debugging). El patrón es
`tape.gradient(loss, vars)` → `optimizer.apply_gradients(zip(grads, vars))`.

Requiere: `tensorflow` / `keras` (≥ 3.0).

## 1. Datos y modelo (MLP sobre Fashion-MNIST vía `tf.data`)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype("float32") / 255.0
X_te = X_te.reshape(-1, 784).astype("float32") / 255.0

train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
            .shuffle(1024).batch(128).prefetch(tf.data.AUTOTUNE))

modelo = keras.Sequential([
    keras.Input((784,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10),                       # logits (sin softmax)
])
print("batches por época:", len(train_ds))

## 2. Loss, optimizer y métricas manuales

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
train_acc = keras.metrics.SparseCategoricalAccuracy()
train_loss = keras.metrics.Mean()
print("loss, optimizer y métricas listos")

## 3. Un `train_step` con `GradientTape`

In [ ]:
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = modelo(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, modelo.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo.trainable_variables))
    train_loss.update_state(loss)
    train_acc.update_state(y, logits)
    return loss

xb, yb = next(iter(train_ds))
print("loss de un batch:", float(train_step(xb, yb)))

## 4. Loop completo por épocas y batches

In [ ]:
EPOCAS = 3
for epoca in range(EPOCAS):
    train_loss.reset_state()
    train_acc.reset_state()
    for xb, yb in train_ds:
        train_step(xb, yb)
    print(f"época {epoca + 1}: loss={train_loss.result():.4f} "
          f"acc={train_acc.result():.4f}")

## 5. `tape.watch` para tensores que no son `tf.Variable`

In [ ]:
x = tf.constant(3.0)                  # constant, no Variable -> hay que watch
with tf.GradientTape() as tape:
    tape.watch(x)
    y = x ** 3                       # dy/dx = 3x^2 = 27 en x=3
print("dy/dx con tape.watch:", float(tape.gradient(y, x)), "(esperado 27)")

## 6. Acelerar el step con `@tf.function`

In [ ]:
@tf.function
def train_step_rapido(x, y):
    with tf.GradientTape() as tape:
        logits = modelo(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, modelo.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo.trainable_variables))
    return loss

xb, yb = next(iter(train_ds))
print("loss (step compilado):", float(train_step_rapido(xb, yb)))
print("El mismo loop con @tf.function corre 2-10x más rápido.")

## Ejercicios

1. **TF loop manual**: entrená un MLP en Fashion-MNIST con `GradientTape` y
   logging manual de loss y accuracy por época.
2. **PyTorch equivalente**: reimplementá el mismo loop con
   `optimizer.zero_grad(); loss.backward(); optimizer.step()`.
3. **Lightning**: resolvé el mismo problema con un `LightningModule.training_step`.
4. **Speedup con jit**: envolvé el `train_step` con `@tf.function` (TF) y con
   `torch.compile` (PyTorch); medí la mejora.
5. **Multi-optimizer**: escribí un loop que aplique un optimizer con LR bajo a
   las capas viejas y otro con LR alto a las nuevas.

## Conclusiones

- El patrón manual: `with tf.GradientTape() as tape:` → `tape.gradient(loss, vars)` → `optimizer.apply_gradients(zip(grads, vars))`.
- Las métricas se manejan a mano: `update_state` en cada batch y `reset_state` al inicio de cada época.
- `tape.watch(t)` fuerza a registrar tensores que no son `tf.Variable` (constantes).
- En capas con `BatchNorm`/`Dropout` hay que pasar `training=True` explícitamente en el forward.
- Envolver el `train_step` en `@tf.function` da el speedup del grafo sin perder el control del loop.